# 🌲 Masterclass 03: Tree-Based Models & Boosting Ensembles
This notebook details non-parametric recursive splitting and boosting trees:

1. **Project 1 (Scratch)**: A complete recursive Decision Tree Classifier with Gini splits from scratch.
2. **Project 2 (Applied)**: A customer churn engine using LightGBM/XGBoost, optimized via Bayesian Optuna.


## 📐 Part 1: Mathematical Foundations
Decision trees perform sequential rectangular splits on features to maximize class purity.

### 1. Shannon Entropy
$$H(S) = -\sum_{i=1}^{C} p_i \log_2 p_i$$

### 2. Gini Impurity (Preferred for Computational Efficiency)
$$Gini(S) = 1 - \sum_{i=1}^{C} p_i^2$$

### 3. Gradient Boosting Mechanics
Instead of averaging predictors (like Random Forest), Boosting constructs successive trees $f_t(x)$ to minimize the loss residual errors of the previous sequence: $\mathcal{L}^{(t)} \approx \sum [g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i)] + \Omega(f_t)$.


## 🧠 Project 1: Decision Tree Classifier from Scratch


In [ ]:
class DecisionNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None

class DecisionTreeScratch:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def _gini(self, y):
        m = len(y)
        if m == 0: return 0.0
        p = np.bincount(y) / m
        return 1.0 - np.sum(p ** 2)

    def _split(self, X, feature, threshold):
        left_idx = np.where(X[:, feature] <= threshold)[0]
        right_idx = np.where(X[:, feature] > threshold)[0]
        return left_idx, right_idx

    def _best_split(self, X, y):
        best_gain = -1.0
        split_idx, split_thresh = None, None
        current_gini = self._gini(y)
        n_samples, n_features = X.shape

        for feat in range(n_features):
            thresholds = np.unique(X[:, feat])
            for thresh in thresholds:
                left_idx, right_idx = self._split(X, feat, thresh)
                if len(left_idx) == 0 or len(right_idx) == 0: continue

                w_gini = (len(left_idx)/n_samples)*self._gini(y[left_idx]) + (len(right_idx)/n_samples)*self._gini(y[right_idx])
                gain = current_gini - w_gini

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat
                    split_thresh = thresh
        return split_idx, split_thresh

    def _build_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))

        if depth >= self.max_depth or n_samples < self.min_samples_split or n_classes == 1:
            return DecisionNode(value=np.argmax(np.bincount(y)))

        feat, thresh = self._best_split(X, y)
        if feat is None: return DecisionNode(value=np.argmax(np.bincount(y)))

        left_idx, right_idx = self._split(X, feat, thresh)
        left_c = self._build_tree(X[left_idx], y[left_idx], depth + 1)
        right_c = self._build_tree(X[right_idx], y[right_idx], depth + 1)
        return DecisionNode(feature=feat, threshold=thresh, left=left_c, right=right_c)

    def fit(self, X, y):
        self.root = self._build_tree(X, y)

    def _predict_row(self, node, x):
        if node.is_leaf(): return node.value
        if x[node.feature] <= node.threshold: return self._predict_row(node.left, x)
        return self._predict_row(node.right, x)

    def predict(self, X):
        return np.array([self._predict_row(self.root, x) for x in X])


## 🧪 Project 2: High-Performance Churn Prediction with LightGBM & Optuna


In [ ]:
import lightgbm as lgb
import optuna

# Synthetic churn dataset
X_churn = np.random.randn(200, 4)
y_churn = np.random.choice([0, 1], size=200, p=[0.75, 0.25])

def objective(trial):
    params = {
        'objective': 'binary',
        'verbosity': -1,
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1)
    }
    train_set = lgb.Dataset(X_churn, label=y_churn)
    cv_res = lgb.cv(params, train_set, num_boost_round=100, nfold=3)
    return cv_res['valid binary_logloss-mean'][-1]

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)
print('Optimal Parameters:', study.best_params)
